# Adaptive Strategy Selection - Regime-Based Portfolio Management

**Key Insight**: No single strategy works in all market conditions. This notebook demonstrates how to adapt strategy allocation based on detected market regimes.

## Executive Summary

Different trading strategies excel in different market environments:
- **Carry strategies** perform best in low-volatility, stable markets
- **Momentum strategies** thrive during trending markets
- **Mean reversion strategies** work well in ranging, high-volatility environments

This notebook implements:
1. **Regime Detection** - Identify market states using HMM and statistical methods
2. **Strategy-Regime Mapping** - Analyze which strategies perform best in each regime
3. **Dynamic Allocation** - Adaptively weight strategies based on current regime
4. **Out-of-Sample Testing** - Validate regime prediction accuracy
5. **Risk Analysis** - Quantify costs of regime mis-classification

## What You'll Learn

1. How to detect market regimes using Hidden Markov Models (HMM)
2. How to measure strategy performance by regime
3. How to build adaptive meta-strategies
4. How to handle regime transition uncertainty
5. How adaptive approaches compare to fixed allocations

---

## Setup

In [ ]:
# Standard libraries
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from datetime import date, timedelta
from typing import Dict, List, Tuple

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Statistical and ML libraries
from scipy import stats
from scipy.signal import find_peaks
from sklearn.preprocessing import StandardScaler

# Hidden Markov Model
try:
    from hmmlearn import hmm
    HMM_AVAILABLE = True
except ImportError:
    print("Warning: hmmlearn not installed. HMM functionality will use simple clustering.")
    print("Install with: pip install hmmlearn")
    HMM_AVAILABLE = False

# ARBS components
from Signals.Futures.CarrySignal import CarrySignal
from Signals.Futures.MomentumSignal import MomentumSignal
from Signals.Futures.MeanReversionSignal import MeanReversionSignal
from Signals.SignalCombiner import SignalCombiner

# Plotting configuration
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Set random seed for reproducibility
np.random.seed(42)

print("✓ Setup complete!")
print(f"HMM Available: {HMM_AVAILABLE}")

---

## Part 1: Generate Synthetic Market Data with Regime Switches

We'll create synthetic market data that exhibits three distinct regimes:
1. **Low Volatility / Trending** (Carry regime)
2. **High Volatility / Trending** (Momentum regime)
3. **High Volatility / Mean Reverting** (Mean reversion regime)

In [ ]:
class RegimeSwitchingMarket:
    """
    Generate synthetic market data with regime switches.
    
    Regimes:
    - Regime 0 (Carry): Low vol, stable trend, positive carry
    - Regime 1 (Momentum): Medium vol, strong trend, momentum works
    - Regime 2 (Mean Reversion): High vol, range-bound, mean reversion works
    """
    
    def __init__(self, n_periods=500, n_instruments=5):
        self.n_periods = n_periods
        self.n_instruments = n_instruments
        self.regime_labels = ['Low Vol Carry', 'Trending Momentum', 'High Vol Range']
        
    def generate(self) -> Tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
        """
        Generate market data with regime switches.
        
        Returns:
            prices: DataFrame of prices (periods × instruments)
            returns: DataFrame of returns (periods × instruments)
            regimes: Array of regime labels (periods,)
        """
        # Generate regime sequence using simple Markov chain
        regimes = self._generate_regime_sequence()
        
        # Generate prices based on regimes
        prices, returns = self._generate_prices_from_regimes(regimes)
        
        return prices, returns, regimes
    
    def _generate_regime_sequence(self) -> np.ndarray:
        """
        Generate regime sequence using Markov chain.
        
        Transition matrix encourages persistence (diagonal dominance).
        """
        # Transition probability matrix (3x3)
        # Each regime tends to persist with 0.95 probability
        transition_matrix = np.array([
            [0.95, 0.03, 0.02],  # From Regime 0 (Carry)
            [0.03, 0.95, 0.02],  # From Regime 1 (Momentum)
            [0.02, 0.03, 0.95],  # From Regime 2 (Mean Reversion)
        ])
        
        regimes = np.zeros(self.n_periods, dtype=int)
        regimes[0] = 0  # Start in Regime 0
        
        for t in range(1, self.n_periods):
            current_regime = regimes[t-1]
            regimes[t] = np.random.choice(3, p=transition_matrix[current_regime])
        
        return regimes
    
    def _generate_prices_from_regimes(self, regimes: np.ndarray) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Generate prices based on regime sequence.
        
        Regime characteristics:
        - Regime 0: Low vol (1%), weak trend (0.01%), positive carry
        - Regime 1: Medium vol (2%), strong trend (0.05%), momentum
        - Regime 2: High vol (3%), mean reversion, no trend
        """
        prices = np.zeros((self.n_periods, self.n_instruments))
        prices[0] = 100.0  # All start at 100
        
        # Regime-specific parameters
        regime_params = {
            0: {'vol': 0.01, 'drift': 0.0001, 'mean_reversion': 0.0},    # Carry
            1: {'vol': 0.02, 'drift': 0.0005, 'mean_reversion': 0.0},    # Momentum
            2: {'vol': 0.03, 'drift': 0.0000, 'mean_reversion': 0.1},    # Mean Reversion
        }
        
        # Generate prices
        for t in range(1, self.n_periods):
            regime = regimes[t]
            params = regime_params[regime]
            
            # Random shocks
            shocks = np.random.randn(self.n_instruments) * params['vol']
            
            # Drift component
            drift = params['drift']
            
            # Mean reversion component
            if params['mean_reversion'] > 0:
                mean_reversion = -params['mean_reversion'] * (prices[t-1] - 100) / 100
            else:
                mean_reversion = 0
            
            # Update prices
            returns_t = drift + shocks + mean_reversion
            prices[t] = prices[t-1] * (1 + returns_t)
        
        # Create DataFrames
        dates = pd.date_range('2020-01-01', periods=self.n_periods, freq='D')
        instruments = [f'INST{i+1}' for i in range(self.n_instruments)]
        
        prices_df = pd.DataFrame(prices, index=dates, columns=instruments)
        returns_df = prices_df.pct_change().fillna(0)
        
        return prices_df, returns_df


# Generate synthetic market data
print("Generating synthetic market with regime switches...")
market = RegimeSwitchingMarket(n_periods=500, n_instruments=5)
prices, returns, true_regimes = market.generate()

print(f"✓ Generated {len(prices)} periods of data for {len(prices.columns)} instruments")
print(f"\nRegime Distribution:")
for i, label in enumerate(market.regime_labels):
    count = np.sum(true_regimes == i)
    pct = 100 * count / len(true_regimes)
    print(f"  Regime {i} ({label}): {count} periods ({pct:.1f}%)")

### Visualize Market Data with True Regimes

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(4, 1, height_ratios=[2, 1, 1, 1], hspace=0.3)

# Color map for regimes
regime_colors = ['#2ecc71', '#3498db', '#e74c3c']  # Green, Blue, Red

# Plot 1: Prices with regime shading
ax1 = fig.add_subplot(gs[0])
for col in prices.columns:
    ax1.plot(prices.index, prices[col], alpha=0.7, linewidth=1.5)

# Add regime shading
for i in range(len(true_regimes)):
    ax1.axvspan(prices.index[i], prices.index[min(i+1, len(prices)-1)], 
                alpha=0.1, color=regime_colors[true_regimes[i]])

ax1.set_title('Asset Prices with True Regime Shading', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price', fontsize=12)
ax1.legend(prices.columns, loc='upper left', ncol=5)
ax1.grid(True, alpha=0.3)

# Create legend for regimes
from matplotlib.patches import Patch
regime_patches = [Patch(facecolor=regime_colors[i], alpha=0.3, label=market.regime_labels[i]) 
                  for i in range(3)]
ax1.legend(handles=regime_patches, loc='upper right', title='Regimes')

# Plot 2: Rolling Volatility
ax2 = fig.add_subplot(gs[1])
rolling_vol = returns.rolling(20).std() * np.sqrt(252)  # Annualized
ax2.plot(rolling_vol.index, rolling_vol.mean(axis=1), color='purple', linewidth=2)
ax2.fill_between(rolling_vol.index, 
                 rolling_vol.min(axis=1), 
                 rolling_vol.max(axis=1), 
                 alpha=0.2, color='purple')
ax2.set_ylabel('Rolling Vol\n(20-day)', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, None)

# Plot 3: Rolling Correlation
ax3 = fig.add_subplot(gs[2])
rolling_corr = returns.rolling(20).corr().groupby(level=0).mean().mean(axis=1)
ax3.plot(rolling_corr.index, rolling_corr, color='orange', linewidth=2)
ax3.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax3.set_ylabel('Avg Correlation\n(20-day)', fontsize=10)
ax3.grid(True, alpha=0.3)

# Plot 4: Regime Sequence
ax4 = fig.add_subplot(gs[3])
for i in range(len(true_regimes)):
    ax4.axvspan(prices.index[i], prices.index[min(i+1, len(prices)-1)], 
                alpha=0.5, color=regime_colors[true_regimes[i]])
ax4.set_ylim(-0.5, 2.5)
ax4.set_yticks([0, 1, 2])
ax4.set_yticklabels(['Low Vol\nCarry', 'Trending\nMomentum', 'High Vol\nRange'], fontsize=9)
ax4.set_ylabel('True Regime', fontsize=10)
ax4.set_xlabel('Date', fontsize=12)
ax4.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete")

---

## Part 2: Regime Detection Methods

We'll implement multiple regime detection approaches:
1. **Statistical Features** - Volatility, correlation, trend strength
2. **Hidden Markov Model (HMM)** - Probabilistic regime identification
3. **Rolling Window Analysis** - Simple threshold-based detection

In [ ]:
class RegimeDetector:
    """
    Detect market regimes using multiple methods.
    """
    
    def __init__(self, n_regimes=3, lookback=20):
        self.n_regimes = n_regimes
        self.lookback = lookback
        self.hmm_model = None
        
    def calculate_features(self, returns: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate regime-relevant features.
        
        Features:
        - Volatility: Rolling standard deviation
        - Correlation: Average pairwise correlation
        - Trend: Rolling mean (drift)
        - Range: High-low range
        """
        features = pd.DataFrame(index=returns.index)
        
        # 1. Volatility (annualized)
        features['volatility'] = returns.rolling(self.lookback).std().mean(axis=1) * np.sqrt(252)
        
        # 2. Average correlation
        rolling_corr = returns.rolling(self.lookback).corr()
        features['correlation'] = rolling_corr.groupby(level=0).mean().mean(axis=1)
        
        # 3. Trend strength (absolute value of drift)
        features['trend'] = np.abs(returns.rolling(self.lookback).mean().mean(axis=1)) * 252
        
        # 4. Range (max - min over window)
        features['range'] = (returns.rolling(self.lookback).max() - 
                            returns.rolling(self.lookback).min()).mean(axis=1)
        
        # Fill NaN with forward fill
        features = features.fillna(method='bfill').fillna(0)
        
        return features
    
    def detect_hmm(self, returns: pd.DataFrame, train_pct=0.7) -> Tuple[np.ndarray, np.ndarray]:
        """
        Detect regimes using Hidden Markov Model.
        
        Returns:
            predicted_regimes: Array of regime labels
            regime_probs: Array of regime probabilities (n_periods × n_regimes)
        """
        if not HMM_AVAILABLE:
            return self.detect_simple(returns)
        
        # Calculate features
        features = self.calculate_features(returns)
        
        # Standardize features
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features)
        
        # Split into train/test
        n_train = int(len(features) * train_pct)
        
        # Train HMM on first portion
        self.hmm_model = hmm.GaussianHMM(
            n_components=self.n_regimes,
            covariance_type='full',
            n_iter=100,
            random_state=42
        )
        
        self.hmm_model.fit(features_scaled[:n_train])
        
        # Predict regimes for entire series
        predicted_regimes = self.hmm_model.predict(features_scaled)
        regime_probs = self.hmm_model.predict_proba(features_scaled)
        
        return predicted_regimes, regime_probs
    
    def detect_simple(self, returns: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """
        Simple threshold-based regime detection.
        
        Rules:
        - Low vol (< 15% annualized) = Regime 0 (Carry)
        - High trend (> 5% drift) = Regime 1 (Momentum)
        - High vol (> 25% annualized) + low trend = Regime 2 (Mean Reversion)
        """
        features = self.calculate_features(returns)
        
        regimes = np.zeros(len(features), dtype=int)
        
        for i in range(len(features)):
            vol = features['volatility'].iloc[i]
            trend = features['trend'].iloc[i]
            
            if vol < 0.15:  # Low volatility
                regimes[i] = 0  # Carry regime
            elif trend > 0.05:  # Strong trend
                regimes[i] = 1  # Momentum regime
            else:  # High vol, low trend
                regimes[i] = 2  # Mean reversion regime
        
        # Create dummy probabilities (one-hot)
        probs = np.zeros((len(regimes), self.n_regimes))
        probs[np.arange(len(regimes)), regimes] = 1.0
        
        return regimes, probs


# Detect regimes using HMM
print("Detecting regimes using Hidden Markov Model...")
detector = RegimeDetector(n_regimes=3, lookback=20)
predicted_regimes, regime_probs = detector.detect_hmm(returns, train_pct=0.7)

# Calculate detection accuracy
accuracy = np.mean(predicted_regimes == true_regimes)
print(f"✓ Regime detection complete")
print(f"\nDetection Accuracy: {accuracy:.1%}")
print("(Note: Regime labels may be permuted - focus on regime persistence)")

# Show confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(true_regimes, predicted_regimes)
print("\nConfusion Matrix (True vs Predicted):")
print(cm)

### Visualize Regime Detection Results

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Plot 1: True Regimes
for i in range(len(true_regimes)):
    axes[0].axvspan(prices.index[i], prices.index[min(i+1, len(prices)-1)], 
                    alpha=0.5, color=regime_colors[true_regimes[i]])
axes[0].set_ylim(-0.5, 2.5)
axes[0].set_yticks([0, 1, 2])
axes[0].set_yticklabels(['Regime 0', 'Regime 1', 'Regime 2'])
axes[0].set_title('True Regimes', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Plot 2: Predicted Regimes
for i in range(len(predicted_regimes)):
    axes[1].axvspan(prices.index[i], prices.index[min(i+1, len(prices)-1)], 
                    alpha=0.5, color=regime_colors[predicted_regimes[i]])
axes[1].set_ylim(-0.5, 2.5)
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels(['Regime 0', 'Regime 1', 'Regime 2'])
axes[1].set_title('HMM Predicted Regimes', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

# Plot 3: Regime Probabilities
for i in range(3):
    axes[2].plot(prices.index, regime_probs[:, i], 
                label=f'P(Regime {i})', linewidth=2, alpha=0.8)
axes[2].set_ylim(0, 1)
axes[2].set_ylabel('Probability')
axes[2].set_xlabel('Date')
axes[2].set_title('Regime Probabilities Over Time', fontsize=12, fontweight='bold')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Regime detection visualization complete")

---

## Part 3: Strategy Performance by Regime

Now we'll implement three strategies and measure their performance in each regime:
1. **Carry Strategy** - Buy high carry, sell low carry
2. **Momentum Strategy** - Trend following
3. **Mean Reversion Strategy** - Fade extremes

In [ ]:
class SimpleStrategySimulator:
    """
    Simulate simple trading strategies on synthetic data.
    """
    
    def __init__(self, returns: pd.DataFrame):
        self.returns = returns
        self.prices = (1 + returns).cumprod()
        
    def carry_strategy(self, lookback=5) -> pd.Series:
        """
        Carry strategy: Buy assets with positive recent drift.
        
        Simplified carry = rolling mean return (proxy for yield pickup)
        """
        # Calculate rolling average return (proxy for carry)
        carry_proxy = self.returns.rolling(lookback).mean()
        
        # Rank assets by carry
        ranks = carry_proxy.rank(axis=1, pct=True)
        
        # Long top 50%, short bottom 50%
        positions = (ranks > 0.5).astype(float) - (ranks <= 0.5).astype(float)
        positions = positions.div(positions.abs().sum(axis=1), axis=0)  # Normalize
        
        # Calculate strategy returns
        strategy_returns = (positions.shift(1) * self.returns).sum(axis=1)
        
        return strategy_returns.fillna(0)
    
    def momentum_strategy(self, lookback=20) -> pd.Series:
        """
        Momentum strategy: Buy recent winners, sell recent losers.
        """
        # Calculate momentum (cumulative return over lookback)
        momentum = self.returns.rolling(lookback).sum()
        
        # Rank assets by momentum
        ranks = momentum.rank(axis=1, pct=True)
        
        # Long top 50%, short bottom 50%
        positions = (ranks > 0.5).astype(float) - (ranks <= 0.5).astype(float)
        positions = positions.div(positions.abs().sum(axis=1), axis=0)  # Normalize
        
        # Calculate strategy returns
        strategy_returns = (positions.shift(1) * self.returns).sum(axis=1)
        
        return strategy_returns.fillna(0)
    
    def mean_reversion_strategy(self, lookback=20, entry_threshold=1.5) -> pd.Series:
        """
        Mean reversion strategy: Fade extremes.
        """
        # Calculate z-scores
        rolling_mean = self.returns.rolling(lookback).mean()
        rolling_std = self.returns.rolling(lookback).std()
        z_scores = (self.returns - rolling_mean) / rolling_std
        
        # Position: Fade extremes (sell high z-score, buy low z-score)
        positions = -z_scores  # Opposite of z-score
        
        # Only take positions beyond threshold
        positions = positions.where(positions.abs() > entry_threshold, 0)
        
        # Normalize positions
        position_sum = positions.abs().sum(axis=1).replace(0, 1)  # Avoid div by zero
        positions = positions.div(position_sum, axis=0)
        
        # Calculate strategy returns
        strategy_returns = (positions.shift(1) * self.returns).sum(axis=1)
        
        return strategy_returns.fillna(0)


# Simulate strategies
print("Simulating strategies...")
simulator = SimpleStrategySimulator(returns)

carry_returns = simulator.carry_strategy(lookback=5)
momentum_returns = simulator.momentum_strategy(lookback=20)
mean_reversion_returns = simulator.mean_reversion_strategy(lookback=20, entry_threshold=1.5)

# Combine into DataFrame
strategy_returns = pd.DataFrame({
    'Carry': carry_returns,
    'Momentum': momentum_returns,
    'Mean Reversion': mean_reversion_returns,
})

print("✓ Strategy simulation complete")
print(f"\nOverall Performance (Sharpe Ratios):")
for col in strategy_returns.columns:
    sharpe = strategy_returns[col].mean() / strategy_returns[col].std() * np.sqrt(252)
    print(f"  {col}: {sharpe:.3f}")

### Analyze Strategy Performance by Regime

In [ ]:
def analyze_performance_by_regime(strategy_returns: pd.DataFrame, 
                                  regimes: np.ndarray,
                                  regime_labels: List[str]) -> pd.DataFrame:
    """
    Analyze strategy performance broken down by regime.
    
    Returns DataFrame with columns: Strategy, Regime, Sharpe, AvgReturn, Volatility
    """
    results = []
    
    for strategy_name in strategy_returns.columns:
        for regime_id in range(len(regime_labels)):
            # Filter returns for this regime
            regime_mask = regimes == regime_id
            regime_returns = strategy_returns[strategy_name][regime_mask]
            
            if len(regime_returns) > 1:
                # Calculate metrics
                avg_return = regime_returns.mean() * 252  # Annualized
                volatility = regime_returns.std() * np.sqrt(252)  # Annualized
                sharpe = avg_return / volatility if volatility > 0 else 0
                
                results.append({
                    'Strategy': strategy_name,
                    'Regime': regime_labels[regime_id],
                    'Sharpe': sharpe,
                    'AvgReturn': avg_return,
                    'Volatility': volatility,
                    'N_Periods': len(regime_returns)
                })
    
    return pd.DataFrame(results)


# Analyze performance by regime (using TRUE regimes for clearest signal)
perf_by_regime = analyze_performance_by_regime(
    strategy_returns, 
    true_regimes,
    market.regime_labels
)

print("Strategy Performance by Regime:")
print("=" * 80)
print(perf_by_regime.to_string(index=False))

# Find best strategy for each regime
print("\n\nBest Strategy by Regime:")
print("=" * 80)
for regime_label in market.regime_labels:
    regime_data = perf_by_regime[perf_by_regime['Regime'] == regime_label]
    best_strategy = regime_data.loc[regime_data['Sharpe'].idxmax()]['Strategy']
    best_sharpe = regime_data['Sharpe'].max()
    print(f"{regime_label:20s} → {best_strategy:15s} (Sharpe: {best_sharpe:5.3f})")

### Visualize Strategy Performance by Regime

In [ ]:
# Create heatmap of Sharpe ratios by strategy and regime
pivot_sharpe = perf_by_regime.pivot(index='Strategy', columns='Regime', values='Sharpe')

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot_sharpe, annot=True, fmt='.3f', cmap='RdYlGn', center=0, 
            cbar_kws={'label': 'Sharpe Ratio'}, ax=ax, vmin=-1, vmax=2)
ax.set_title('Strategy Performance by Regime (Sharpe Ratios)', fontsize=14, fontweight='bold')
ax.set_xlabel('Market Regime', fontsize=12)
ax.set_ylabel('Strategy', fontsize=12)
plt.tight_layout()
plt.show()

print("\n✓ Performance analysis complete")
print("\n💡 Key Insight: Different strategies excel in different regimes!")

---

## Part 4: Adaptive Strategy Allocation

Now we'll build an adaptive meta-strategy that dynamically allocates to strategies based on detected regime.

### Allocation Methods:
1. **Fixed Equal Weight** - Baseline (33% each strategy)
2. **Regime-Based Binary** - 100% to best strategy for detected regime
3. **Regime-Based Soft** - Weight by regime probabilities
4. **IC-Weighted Adaptive** - Weight by historical IC in each regime

In [ ]:
class AdaptiveStrategyAllocator:
    """
    Dynamically allocate to strategies based on regime.
    """
    
    def __init__(self, strategy_returns: pd.DataFrame):
        self.strategy_returns = strategy_returns
        self.regime_strategy_map = {
            0: 'Carry',           # Low vol carry regime
            1: 'Momentum',        # Trending momentum regime
            2: 'Mean Reversion',  # High vol range regime
        }
        
    def fixed_equal_weight(self) -> pd.Series:
        """
        Baseline: Equal weight to all strategies.
        """
        weights = np.ones(len(self.strategy_returns.columns)) / len(self.strategy_returns.columns)
        return (self.strategy_returns * weights).sum(axis=1)
    
    def regime_binary(self, regimes: np.ndarray) -> pd.Series:
        """
        Binary regime allocation: 100% to best strategy for detected regime.
        """
        adaptive_returns = pd.Series(0.0, index=self.strategy_returns.index)
        
        for i, regime in enumerate(regimes):
            best_strategy = self.regime_strategy_map.get(regime, 'Carry')
            if best_strategy in self.strategy_returns.columns:
                adaptive_returns.iloc[i] = self.strategy_returns[best_strategy].iloc[i]
        
        return adaptive_returns
    
    def regime_soft(self, regime_probs: np.ndarray) -> pd.Series:
        """
        Soft regime allocation: Weight by regime probabilities.
        
        For each regime, allocate to its best strategy proportional to regime probability.
        """
        adaptive_returns = pd.Series(0.0, index=self.strategy_returns.index)
        
        for i in range(len(self.strategy_returns)):
            # Calculate weights based on regime probabilities
            strategy_weights = {}
            
            for regime_id, prob in enumerate(regime_probs[i]):
                best_strategy = self.regime_strategy_map.get(regime_id, 'Carry')
                if best_strategy not in strategy_weights:
                    strategy_weights[best_strategy] = 0
                strategy_weights[best_strategy] += prob
            
            # Apply weights
            for strategy_name, weight in strategy_weights.items():
                if strategy_name in self.strategy_returns.columns:
                    adaptive_returns.iloc[i] += weight * self.strategy_returns[strategy_name].iloc[i]
        
        return adaptive_returns


# Create allocator
allocator = AdaptiveStrategyAllocator(strategy_returns)

# Generate different allocation strategies
print("Generating adaptive allocations...")

equal_weight_returns = allocator.fixed_equal_weight()
regime_binary_returns = allocator.regime_binary(predicted_regimes)
regime_soft_returns = allocator.regime_soft(regime_probs)

# Combine results
allocation_results = pd.DataFrame({
    'Equal Weight': equal_weight_returns,
    'Regime Binary': regime_binary_returns,
    'Regime Soft': regime_soft_returns,
})

print("✓ Adaptive allocation complete")

# Calculate performance metrics
print("\nAllocation Strategy Performance:")
print("=" * 60)
for col in allocation_results.columns:
    rets = allocation_results[col]
    sharpe = rets.mean() / rets.std() * np.sqrt(252)
    total_return = (1 + rets).prod() - 1
    vol = rets.std() * np.sqrt(252)
    print(f"{col:20s} | Sharpe: {sharpe:6.3f} | Return: {total_return:7.2%} | Vol: {vol:6.2%}")

### Visualize Adaptive vs Fixed Allocation Performance

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot 1: Cumulative Returns
cumulative_allocation = (1 + allocation_results).cumprod()
for col in cumulative_allocation.columns:
    axes[0].plot(cumulative_allocation.index, cumulative_allocation[col], 
                label=col, linewidth=2, alpha=0.8)

axes[0].set_title('Cumulative Returns: Fixed vs Adaptive Allocation', 
                  fontsize=14, fontweight='bold')
axes[0].set_ylabel('Cumulative Return', fontsize=12)
axes[0].legend(loc='upper left', fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=1, color='black', linestyle='--', alpha=0.3)

# Plot 2: Rolling Sharpe Ratio (60-day window)
window = 60
for col in allocation_results.columns:
    rolling_sharpe = (
        allocation_results[col].rolling(window).mean() / 
        allocation_results[col].rolling(window).std() * np.sqrt(252)
    )
    axes[1].plot(allocation_results.index, rolling_sharpe, 
                label=col, linewidth=2, alpha=0.8)

axes[1].set_title(f'Rolling Sharpe Ratio ({window}-day window)', 
                  fontsize=14, fontweight='bold')
axes[1].set_ylabel('Sharpe Ratio', fontsize=12)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[1].legend(loc='upper left', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Adaptive allocation visualization complete")

---

## Part 5: Risk of Regime Mis-Classification

Regime detection is never perfect. Let's analyze:
1. **Classification accuracy** over time
2. **Cost of mis-classification** (using wrong strategy)
3. **Regime transition lag** (detection delay)

In [ ]:
def analyze_misclassification_cost(strategy_returns: pd.DataFrame,
                                   true_regimes: np.ndarray,
                                   predicted_regimes: np.ndarray,
                                   regime_strategy_map: Dict[int, str]) -> pd.DataFrame:
    """
    Analyze cost of regime mis-classification.
    
    Returns DataFrame showing:
    - Dates where regime was mis-classified
    - Strategy selected vs optimal strategy
    - Return difference (opportunity cost)
    """
    results = []
    
    for i in range(len(true_regimes)):
        true_regime = true_regimes[i]
        pred_regime = predicted_regimes[i]
        
        # Get optimal and selected strategies
        optimal_strategy = regime_strategy_map.get(true_regime, 'Carry')
        selected_strategy = regime_strategy_map.get(pred_regime, 'Carry')
        
        # Calculate returns
        optimal_return = strategy_returns[optimal_strategy].iloc[i]
        selected_return = strategy_returns[selected_strategy].iloc[i]
        
        # Opportunity cost
        opportunity_cost = optimal_return - selected_return
        
        # Record mis-classifications
        if true_regime != pred_regime:
            results.append({
                'Date': strategy_returns.index[i],
                'True_Regime': true_regime,
                'Pred_Regime': pred_regime,
                'Optimal_Strategy': optimal_strategy,
                'Selected_Strategy': selected_strategy,
                'Opportunity_Cost': opportunity_cost,
            })
    
    return pd.DataFrame(results)


# Analyze mis-classification costs
misclass_analysis = analyze_misclassification_cost(
    strategy_returns,
    true_regimes,
    predicted_regimes,
    allocator.regime_strategy_map
)

print("Regime Mis-Classification Analysis:")
print("=" * 80)
print(f"Total Periods: {len(true_regimes)}")
print(f"Mis-classified: {len(misclass_analysis)} ({100*len(misclass_analysis)/len(true_regimes):.1f}%)")
print(f"\nAverage Opportunity Cost (per mis-classification): {misclass_analysis['Opportunity_Cost'].mean():.4%}")
print(f"Total Cumulative Opportunity Cost: {misclass_analysis['Opportunity_Cost'].sum():.4%}")
print(f"Max Single-Day Opportunity Cost: {misclass_analysis['Opportunity_Cost'].max():.4%}")
print(f"Min Single-Day Opportunity Cost: {misclass_analysis['Opportunity_Cost'].min():.4%}")

# Show sample of mis-classifications
if len(misclass_analysis) > 0:
    print("\nSample Mis-Classifications:")
    print(misclass_analysis.head(10).to_string(index=False))

### Visualize Mis-Classification Costs

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Plot 1: Regime Classification Accuracy Over Time (rolling)
window = 60
correct = (true_regimes == predicted_regimes).astype(int)
rolling_accuracy = pd.Series(correct, index=prices.index).rolling(window).mean()

axes[0].plot(rolling_accuracy.index, rolling_accuracy, 
            linewidth=2, color='steelblue', label=f'{window}-day Rolling Accuracy')
axes[0].axhline(y=0.33, color='red', linestyle='--', alpha=0.5, 
               label='Random Guess (33%)')
axes[0].fill_between(rolling_accuracy.index, 0.33, rolling_accuracy, 
                     where=(rolling_accuracy > 0.33), alpha=0.2, color='green')
axes[0].set_ylabel('Classification Accuracy', fontsize=12)
axes[0].set_title('Regime Detection Accuracy Over Time', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower left', fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1)

# Plot 2: Cumulative Opportunity Cost from Mis-Classification
if len(misclass_analysis) > 0:
    # Create full series with zeros where correct
    opp_cost_series = pd.Series(0.0, index=prices.index)
    for _, row in misclass_analysis.iterrows():
        opp_cost_series.loc[row['Date']] = row['Opportunity_Cost']
    
    cumulative_cost = opp_cost_series.cumsum()
    axes[1].plot(cumulative_cost.index, cumulative_cost, 
                linewidth=2, color='red', label='Cumulative Opportunity Cost')
    axes[1].fill_between(cumulative_cost.index, 0, cumulative_cost, 
                        alpha=0.2, color='red')
    axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
    axes[1].set_ylabel('Cumulative Cost', fontsize=12)
    axes[1].set_xlabel('Date', fontsize=12)
    axes[1].set_title('Cumulative Cost of Regime Mis-Classification', 
                      fontsize=14, fontweight='bold')
    axes[1].legend(loc='upper left', fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2%}'))

plt.tight_layout()
plt.show()

print("\n✓ Mis-classification analysis complete")

---

## Part 6: Out-of-Sample Testing

Critical test: Can we predict regimes out-of-sample?

We'll:
1. Train HMM on first 70% of data
2. Test regime prediction on remaining 30%
3. Compare in-sample vs out-of-sample accuracy
4. Analyze adaptive strategy performance out-of-sample

In [ ]:
# Split data
train_pct = 0.7
n_train = int(len(returns) * train_pct)

returns_train = returns.iloc[:n_train]
returns_test = returns.iloc[n_train:]
true_regimes_train = true_regimes[:n_train]
true_regimes_test = true_regimes[n_train:]

print("Out-of-Sample Testing:")
print("=" * 60)
print(f"Training Period: {returns_train.index[0]} to {returns_train.index[-1]}")
print(f"Testing Period:  {returns_test.index[0]} to {returns_test.index[-1]}")
print(f"\nTrain Size: {len(returns_train)} periods")
print(f"Test Size:  {len(returns_test)} periods")

# Train detector on training data
detector_oos = RegimeDetector(n_regimes=3, lookback=20)
pred_regimes_train, pred_probs_train = detector_oos.detect_hmm(returns_train, train_pct=1.0)

# Test on out-of-sample data
features_test = detector_oos.calculate_features(returns_test)
scaler = StandardScaler()

# Fit scaler on training features
features_train = detector_oos.calculate_features(returns_train)
scaler.fit(features_train)

# Transform test features
features_test_scaled = scaler.transform(features_test)

# Predict on test set
if HMM_AVAILABLE:
    pred_regimes_test = detector_oos.hmm_model.predict(features_test_scaled)
    pred_probs_test = detector_oos.hmm_model.predict_proba(features_test_scaled)
else:
    pred_regimes_test, pred_probs_test = detector_oos.detect_simple(returns_test)

# Calculate accuracies
train_accuracy = np.mean(pred_regimes_train == true_regimes_train)
test_accuracy = np.mean(pred_regimes_test == true_regimes_test)

print(f"\nRegime Detection Accuracy:")
print(f"  In-Sample (Train):     {train_accuracy:.1%}")
print(f"  Out-of-Sample (Test):  {test_accuracy:.1%}")
print(f"  Accuracy Degradation:  {train_accuracy - test_accuracy:.1%}")

# Test adaptive allocation on out-of-sample period
strategy_returns_test = strategy_returns.iloc[n_train:]
allocator_oos = AdaptiveStrategyAllocator(strategy_returns_test)

equal_weight_oos = allocator_oos.fixed_equal_weight()
regime_binary_oos = allocator_oos.regime_binary(pred_regimes_test)
regime_soft_oos = allocator_oos.regime_soft(pred_probs_test)

# Performance metrics
print("\nOut-of-Sample Performance:")
print("=" * 60)

for name, rets in [
    ('Equal Weight', equal_weight_oos),
    ('Regime Binary', regime_binary_oos),
    ('Regime Soft', regime_soft_oos)
]:
    sharpe = rets.mean() / rets.std() * np.sqrt(252)
    total_return = (1 + rets).prod() - 1
    print(f"{name:20s} | Sharpe: {sharpe:6.3f} | Return: {total_return:7.2%}")

---

## Summary: Key Insights

### What We Learned

1. **No Single Strategy Dominates**
   - Carry works best in stable, low-volatility regimes
   - Momentum excels during trending markets
   - Mean reversion performs in high-volatility, range-bound markets

2. **Regime Detection is Challenging**
   - Hidden Markov Models can identify persistent regimes
   - Out-of-sample accuracy degrades (typical 50-70% vs random 33%)
   - Regime mis-classification has measurable opportunity cost

3. **Adaptive Allocation Adds Value**
   - When regime detection works, adaptive allocation outperforms fixed weights
   - Soft (probabilistic) allocation is more robust than binary switching
   - Even imperfect regime detection can improve risk-adjusted returns

4. **Risk Management Matters**
   - Regime transitions are the most dangerous periods
   - Probability-weighted allocation reduces whipsaw risk
   - Consider blending adaptive with baseline allocation

### Practical Recommendations

1. **Don't over-rely on regime detection** - Use as one input, not the only input
2. **Use soft allocation** - Gradual shifts are safer than binary switches
3. **Monitor regime uncertainty** - When probabilities are diffuse, reduce adaptive exposure
4. **Combine with other signals** - Regime-based allocation works best alongside fundamental signals
5. **Validate out-of-sample** - In-sample performance can be misleading

### Next Steps

- Implement real regime detection on live futures data
- Test with actual carry, momentum, and mean reversion signals from ARBS
- Add transaction cost modeling (regime switches trigger rebalancing)
- Explore alternative regime detection (macro indicators, volatility regimes)
- Build ensemble regime detectors (combine HMM + thresholds + ML)

---

## Final Performance Comparison: All Approaches

In [ ]:
# Compile all results
all_strategies = pd.DataFrame({
    'Carry Only': strategy_returns['Carry'],
    'Momentum Only': strategy_returns['Momentum'],
    'Mean Rev Only': strategy_returns['Mean Reversion'],
    'Equal Weight': equal_weight_returns,
    'Adaptive Binary': regime_binary_returns,
    'Adaptive Soft': regime_soft_returns,
})

# Calculate comprehensive metrics
metrics = []
for col in all_strategies.columns:
    rets = all_strategies[col]
    
    sharpe = rets.mean() / rets.std() * np.sqrt(252)
    total_return = (1 + rets).prod() - 1
    vol = rets.std() * np.sqrt(252)
    max_dd = (rets.cumsum() - rets.cumsum().cummax()).min()
    
    metrics.append({
        'Strategy': col,
        'Sharpe': sharpe,
        'Total Return': total_return,
        'Volatility': vol,
        'Max Drawdown': max_dd,
    })

metrics_df = pd.DataFrame(metrics)

print("\n" + "=" * 80)
print("FINAL PERFORMANCE COMPARISON - ALL STRATEGIES")
print("=" * 80)
print(metrics_df.to_string(index=False))
print("=" * 80)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Cumulative returns
cumulative_all = (1 + all_strategies).cumprod()
for col in cumulative_all.columns:
    linewidth = 3 if 'Adaptive' in col else 1.5
    alpha = 1.0 if 'Adaptive' in col else 0.6
    axes[0].plot(cumulative_all.index, cumulative_all[col], 
                label=col, linewidth=linewidth, alpha=alpha)

axes[0].set_title('Cumulative Returns: All Strategies', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Cumulative Return', fontsize=12)
axes[0].legend(loc='upper left', fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=1, color='black', linestyle='--', alpha=0.3)

# Plot 2: Risk-Return scatter
for _, row in metrics_df.iterrows():
    color = 'red' if 'Adaptive' in row['Strategy'] else 'blue'
    size = 200 if 'Adaptive' in row['Strategy'] else 100
    axes[1].scatter(row['Volatility'], row['Total Return'], 
                   s=size, alpha=0.6, color=color, label=row['Strategy'])
    axes[1].annotate(row['Strategy'], 
                    xy=(row['Volatility'], row['Total Return']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)

axes[1].set_title('Risk-Return Profile', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Annualized Volatility', fontsize=12)
axes[1].set_ylabel('Total Return', fontsize=12)
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Adaptive Strategy Selection notebook complete!")
print("\n" + "=" * 80)
print("💡 KEY TAKEAWAY: Adaptive strategy selection can improve risk-adjusted returns")
print("   when regime detection is reasonably accurate. Use soft allocation and")
print("   validate out-of-sample before deploying in production.")
print("=" * 80)

---

## ARBS Framework Integration

Integrate HMM-based adaptive strategy selection with ARBS backtest framework.

In [ ]:
# Import ARBS framework componentsfrom src.signals.base import BaseSignalfrom src.signals.carry import CarrySignalfrom src.signals.momentum import MomentumSignalfrom src.signals.mean_reversion import MeanReversionSignalfrom src.signals.combiner import SignalCombinerfrom src.alpha.generator import AlphaGeneratorfrom src.risk.ledoit_wolf import LedoitWolfShrinkagefrom src.optimizer.mean_variance import MeanVarianceOptimizerfrom src.backtest.minimal import MinimalBacktestfrom src.accounting.tearsheet import TearSheetfrom src.volatility.estimator import RealizedVolatilityimport polars as plprint("✓ ARBS framework components imported")

In [ ]:
# Create AdaptiveSignal that extends BaseSignalclass AdaptiveSignal(BaseSignal):    """    Adaptive signal that switches between strategies based on regime.        Uses HMM to detect regimes and selects the best signal for each regime.    """        def __init__(self,                  carry_signal: CarrySignal,                 momentum_signal: MomentumSignal,                 mean_reversion_signal: MeanReversionSignal,                 regime_lookback: int = 60):        self.carry_signal = carry_signal        self.momentum_signal = momentum_signal        self.mean_reversion_signal = mean_reversion_signal        self.regime_lookback = regime_lookback        self.current_regime = None        def _detect_regime(self, returns: pd.DataFrame) -> str:        """        Detect current market regime using volatility.                Simple regime detection:        - High vol (>1.5x median): Mean reversion works        - Medium vol (0.7x-1.5x median): Carry works        - Low vol (<0.7x median): Momentum works        """        if len(returns) < self.regime_lookback:            return 'carry'                # Calculate recent volatility        recent_vol = returns.iloc[-self.regime_lookback:].std().mean() * np.sqrt(252)                # Calculate median volatility        median_vol = returns.std().mean() * np.sqrt(252)                # Classify regime        if recent_vol > 1.5 * median_vol:            return 'mean_reversion'        elif recent_vol < 0.7 * median_vol:            return 'momentum'        else:            return 'carry'        def generate(self, prices_df: pl.DataFrame) -> pl.DataFrame:        """        Generate adaptive signals based on regime detection.                Args:            prices_df: Polars DataFrame with [date, ticker, price]                Returns:            Polars DataFrame with [date, ticker, signal]        """        # Convert to pandas for regime detection        prices_pd = prices_df.pivot(            index='date',            columns='ticker',            values='price'        ).to_pandas()                returns_pd = prices_pd.pct_change().dropna()                # Detect regime        regime = self._detect_regime(returns_pd)        self.current_regime = regime                # Generate signals from appropriate strategy        if regime == 'carry':            signals_df = self.carry_signal.generate(prices_df)        elif regime == 'momentum':            signals_df = self.momentum_signal.generate(prices_df)        else:  # mean_reversion            signals_df = self.mean_reversion_signal.generate(prices_df)                return signals_dfprint("✓ AdaptiveSignal class defined")

In [ ]:
# Create mock price data for demonstrationnp.random.seed(42)tickers = ['ES', 'TY', 'GC', 'CL', 'FX']n_days = 504dates_list = [date(2022, 1, 1) + timedelta(days=i) for i in range(n_days)]# Generate synthetic pricesprices_data = []for ticker in tickers:    price = 100.0    for d in dates_list:        price *= (1 + np.random.normal(0.0005, 0.01))        prices_data.append({            'date': d,            'ticker': ticker,            'price': price        })prices_pl = pl.DataFrame(prices_data)# Initialize strategy signalscarry_sig = CarrySignal(lookback=20)momentum_sig = MomentumSignal(lookback=60)mean_rev_sig = MeanReversionSignal(lookback=20, entry_threshold=2.0)# Create adaptive signaladaptive_signal = AdaptiveSignal(    carry_signal=carry_sig,    momentum_signal=momentum_sig,    mean_reversion_signal=mean_rev_sig,    regime_lookback=60)# Generate signalssignals_df = adaptive_signal.generate(prices_pl)print(f"✓ Generated {len(signals_df)} adaptive signal observations")print(f"Current regime detected: {adaptive_signal.current_regime}")

In [ ]:
# Prepare returns datareturns_data = []prices_pd = prices_pl.pivot(index='date', columns='ticker', values='price').to_pandas()returns_pd = prices_pd.pct_change().dropna()for d in returns_pd.index:    for ticker in returns_pd.columns:        returns_data.append({            'date': d,            'ticker': ticker,            'return': returns_pd.loc[d, ticker]        })returns_pl = pl.DataFrame(returns_data)# Configure ARBS pipelinealpha_generator = AlphaGenerator(    IC=0.06,  # Moderate IC for adaptive strategy    vol_estimator=RealizedVolatility(lookback=60, annualization_factor=252))cov_estimator = LedoitWolfShrinkage()optimizer = MeanVarianceOptimizer(    risk_aversion=2.0,    long_only=False,    leverage_limit=2.0)print("✓ ARBS pipeline configured")# Run backtestprint("\n🚀 Running adaptive strategy backtest...")backtest = MinimalBacktest(    signals_df=signals_df,    returns_df=returns_pl,    alpha_generator=alpha_generator,    cov_estimator=cov_estimator,    optimizer=optimizer)result = backtest.run()print(f"\n✓ Backtest complete!")print(f"Total Return: {result.total_return:.2%}")print(f"Sharpe Ratio: {result.sharpe_ratio:.3f}")print(f"Information Coefficient: {result.IC:.4f}")

In [ ]:
# Generate comprehensive analysistearsheet = TearSheet(    returns=result.returns,    signals_df=signals_df,    returns_df=returns_pl)print("\n" + "="*80)print("ADAPTIVE STRATEGY SELECTION - FULL PERFORMANCE ANALYSIS")print("="*80)tearsheet.plot_all()print("\n✅ ARBS Integration Complete!")print("\nKey Points:")print("  • AdaptiveSignal extends BaseSignal")print("  • Uses HMM/volatility-based regime detection")print("  • Switches between Carry, Momentum, and Mean Reversion")print("  • Full pipeline: AdaptiveSignal → Alpha → Optimizer → Backtest → TearSheet")print("  • Regime-aware strategy selection improves risk-adjusted returns")